# Leakage-safe CKD screening-opportunity comparison

This educational analysis compares a prevalence/no-skill reference, logistic regression, and a random forest using variables plausibly available before kidney laboratory results. It is **not a diagnosis, patient-targeting system, or clinical decision tool**. The outcome is a cross-sectional laboratory indicator; NHANES cannot establish chronicity.

The strict feature allowlist is age, sex, and race/ethnicity because those are the only defensible pre-laboratory predictors in the locally verified NHANES snapshot. Creatinine, eGFR, UACR, urine albumin/creatinine, indicator fields, and transformed or proxy outcome fields are forbidden. A grouped split keeps NHANES masked survey clusters disjoint. Logistic regression is the pre-specified interpretable primary model before holdout evaluation. Its reference threshold comes from out-of-fold development predictions; the capacity-matched holdout table ranks each model without using holdout labels and uses a deterministic tie policy. Metrics are unweighted analytic-sample performance, not U.S. population-performance estimates.

In [1]:
import json
from dataclasses import asdict
from hashlib import sha256
from pathlib import Path

import pandas as pd

from ckd_intelligence.analysis.artifacts import patient_finding_output_paths
from ckd_intelligence.cohorts.definitions import CKDDefinition, classify_ckd
from ckd_intelligence.modeling.patient_finding import train_patient_finding_models

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data/raw/nhanes/2017-2018'
MANIFEST_PATH = ROOT / 'data/manifests/nhanes-2017-2018-patient-need.json'
FIXTURE_PATH = ROOT / 'data/fixtures/nhanes_patient_need_representative.csv'
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))

def checksum(path):
    return sha256(path.read_bytes()).hexdigest()


In [2]:
expected = {item['name']: item for item in manifest['files']}
has_official_snapshot = all((RAW_DIR / name).is_file() for name in expected)
if has_official_snapshot:
    for name, metadata in expected.items():
        path = RAW_DIR / name
        if path.stat().st_size != metadata['bytes']:
            raise ValueError(f'Unexpected byte count: {name}')
        if checksum(path) != metadata['sha256']:
            raise ValueError(f'Checksum mismatch: {name}')
    demo = pd.read_sas(RAW_DIR / 'DEMO_J.xpt', format='xport')[[
        'SEQN', 'RIDSTATR', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'RIDEXPRG',
        'WTMEC2YR', 'SDMVSTRA', 'SDMVPSU'
    ]]
    biopro = pd.read_sas(RAW_DIR / 'BIOPRO_J.xpt', format='xport')[[
        'SEQN', 'LBXSCR'
    ]]
    albumin = pd.read_sas(RAW_DIR / 'ALB_CR_J.xpt', format='xport')[[
        'SEQN', 'URXUMA', 'URXUCR', 'URDACT'
    ]]
    joined = demo.merge(biopro, on='SEQN', how='left', validate='one_to_one').merge(
        albumin, on='SEQN', how='left', validate='one_to_one'
    )
    people = joined.rename(columns={
        'RIDAGEYR': 'age_years', 'RIDEXPRG': 'pregnancy_status_code',
        'LBXSCR': 'serum_creatinine_mg_dl', 'URXUMA': 'urine_albumin_mg_l',
        'URXUCR': 'urine_creatinine_mg_dl', 'URDACT': 'uacr_mg_g',
        'WTMEC2YR': 'sample_weight', 'SDMVSTRA': 'strata', 'SDMVPSU': 'psu'
    })
    people['sex'] = joined['RIAGENDR'].map({1.0: 'Male', 2.0: 'Female'})
    people['race_ethnicity'] = joined['RIDRETH3'].map({
        1.0: 'Mexican American', 2.0: 'Other Hispanic',
        3.0: 'Non-Hispanic White', 4.0: 'Non-Hispanic Black',
        6.0: 'Non-Hispanic Asian', 7.0: 'Other Race - Including Multi-Racial'
    })
    people.loc[people['sample_weight'].le(1e-12), 'sample_weight'] = pd.NA
    people = people.loc[
        people['RIDSTATR'].eq(2) & people['sample_weight'].notna()
        & people['strata'].notna() & people['psu'].notna()
    ].copy()
    evidence_type = 'public_observed'
    source_label = 'Verified CDC/NCHS NHANES 2017-2018 public-use files'
else:
    base = pd.read_csv(FIXTURE_PATH)
    expanded = []
    races = ['Fixture Group A', 'Fixture Group B', 'Fixture Group C']
    for replica in range(12):
        part = base.copy()
        part['age_years'] = (part['age_years'] + (replica % 5) - 2).clip(18, 85)
        part['race_ethnicity'] = [races[(i + replica) % 3] for i in range(len(part))]
        part['strata'] = replica + 1
        part['psu'] = 1
        expanded.append(part)
    people = pd.concat(expanded, ignore_index=True)
    evidence_type = 'fixture_only'
    source_label = 'Deterministic expanded fixture; not a population finding'

print({'evidence_type': evidence_type, 'source': source_label, 'candidate_rows': len(people)})


{'evidence_type': 'public_observed', 'source': 'Verified CDC/NCHS NHANES 2017-2018 public-use files', 'candidate_rows': 8704}


In [3]:
definitions = [CKDDefinition.primary(), CKDDefinition.egfr_only(), CKDDefinition.albuminuria_only()]
for definition in definitions:
    people[definition.name] = classify_ckd(people, definition)
adult = people['age_years'].ge(18)
known_pregnant = pd.to_numeric(people['pregnancy_status_code'], errors='coerce').eq(1).fillna(False)
complete = people[CKDDefinition.primary().name].notna()
analytic = people.loc[adult & ~known_pregnant & complete, [
    'age_years', 'sex', 'race_ethnicity', 'strata', 'psu',
    CKDDefinition.primary().name, CKDDefinition.egfr_only().name,
    CKDDefinition.albuminuria_only().name
]].copy()
analytic['ckd_indicator'] = analytic[CKDDefinition.primary().name].astype(int)
groups = (
    analytic['strata'].astype(int).astype(str)
    + '-psu-'
    + analytic['psu'].astype(int).astype(str)
)
model_frame = analytic[[
    'age_years', 'sex', 'race_ethnicity', 'ckd_indicator',
    CKDDefinition.egfr_only().name, CKDDefinition.albuminuria_only().name
]]
bootstrap_replicates = 1000 if evidence_type == 'public_observed' else 200
comparison = train_patient_finding_models(
    model_frame,
    groups,
    capacity=0.10,
    bootstrap_replicates=bootstrap_replicates,
)
print({
    'analytic_n': len(model_frame), 'features': list(comparison.features),
    'development_n': comparison.split.development_n, 'holdout_n': comparison.holdout_n,
    'threshold_source': comparison.split.threshold_source,
    'operational_capacity': comparison.threshold_capacity,
    'development_requested_count': comparison.development_capacity.requested_count,
    'development_selected_count': comparison.development_capacity.selected_count,
    'development_selected_share': comparison.development_capacity.selected_share,
    'selection_policy': comparison.selection_policy,
})


{'analytic_n': 5016, 'features': ['age_years', 'sex', 'race_ethnicity'], 'development_n': 4023, 'holdout_n': 993, 'threshold_source': 'out_of_fold_development_predictions', 'operational_capacity': 0.1, 'development_requested_count': 402, 'development_selected_count': 402, 'development_selected_share': 0.09992542878448919, 'selection_policy': 'logistic_prespecified_before_holdout'}


In [4]:
for name, metric in comparison.models.items():
    print(name, {
        'roc_auc': round(metric.roc_auc, 4), 'pr_auc': round(metric.pr_auc, 4),
        'brier_score': round(metric.brier_score, 4),
        'precision_at_capacity_threshold': round(metric.precision, 4),
        'recall_at_capacity_threshold': round(metric.recall, 4),
        'realized_capacity_threshold': round(metric.threshold, 4),
        'requested_count': metric.requested_count,
        'selected_count': metric.selected_count,
        'selected_share': round(metric.selected_share, 4),
        'roc_auc_95_interval': [
            round(metric.roc_auc_interval.low, 4),
            round(metric.roc_auc_interval.high, 4),
        ],
        'pr_auc_95_interval': [
            round(metric.pr_auc_interval.low, 4),
            round(metric.pr_auc_interval.high, 4),
        ],
        'brier_95_interval': [
            round(metric.brier_interval.low, 4),
            round(metric.brier_interval.high, 4),
        ],
        'n': metric.n,
        'positives': metric.positives, 'confusion_matrix': metric.confusion_matrix
    })
print('PR-AUC reference is holdout prevalence:', round(comparison.holdout_prevalence, 4))
print(
    'Decision: logistic regression was pre-specified for interpretability; '
    'holdout comparisons describe uncertainty rather than select the winner.'
)


prevalence_no_skill {'roc_auc': 0.5, 'pr_auc': 0.1863, 'brier_score': 0.1516, 'precision_at_capacity_threshold': 0.2222, 'recall_at_capacity_threshold': 0.1189, 'realized_capacity_threshold': 0.1872, 'requested_count': 99, 'selected_count': 99, 'selected_share': 0.0997, 'roc_auc_95_interval': [0.5, 0.5], 'pr_auc_95_interval': [0.1617, 0.21], 'brier_95_interval': [0.1362, 0.1664], 'n': 993, 'positives': 185, 'confusion_matrix': {'true_negative': 731, 'false_positive': 77, 'false_negative': 163, 'true_positive': 22}}
logistic_regression {'roc_auc': 0.755, 'pr_auc': 0.4625, 'brier_score': 0.13, 'precision_at_capacity_threshold': 0.5354, 'recall_at_capacity_threshold': 0.2865, 'realized_capacity_threshold': 0.3822, 'requested_count': 99, 'selected_count': 99, 'selected_share': 0.0997, 'roc_auc_95_interval': [0.7081, 0.7809], 'pr_auc_95_interval': [0.3903, 0.5135], 'brier_95_interval': [0.1196, 0.1388], 'n': 993, 'positives': 185, 'confusion_matrix': {'true_negative': 762, 'false_positive':

In [5]:
print(
    'Calibration bins:',
    [asdict(item) for item in comparison.models['logistic_regression'].calibration_bins],
)
print('Age-band diagnostics:', [asdict(item) for item in comparison.subgroups['age_band']])
print('Sex diagnostics:', [asdict(item) for item in comparison.subgroups['sex']])
print('Cohort-definition sensitivity:', comparison.cohort_sensitivity)
print('Paired model differences:', {
    name: asdict(value) for name, value in comparison.paired_differences.items()
})


Calibration bins: [{'mean_probability': 0.04165000027524008, 'observed_rate': 0.045, 'n': 200}, {'mean_probability': 0.08023465919391677, 'observed_rate': 0.09045226130653267, 'n': 199}, {'mean_probability': 0.14535437340737045, 'observed_rate': 0.15736040609137056, 'n': 197}, {'mean_probability': 0.23469850811468476, 'observed_rate': 0.18592964824120603, 'n': 199}, {'mean_probability': 0.3825258159108139, 'observed_rate': 0.45454545454545453, 'n': 198}]
Age-band diagnostics: [{'group': 'age_band', 'value': '18-39', 'n': 344, 'positives': 18, 'prevalence': 0.05232558139534884, 'precision': None, 'recall': 0.0, 'brier_score': 0.049643774481603266, 'precision_interval': None, 'recall_interval': {'low': 0.0, 'high': 0.0, 'method': 'holdout_cluster_bootstrap_percentile', 'requested_replicates': 1000, 'valid_replicates': 1000, 'cluster_count': 6, 'caveat': 'Only 6 holdout clusters; interval precision is limited.'}, 'brier_interval': {'low': 0.03645755559059979, 'high': 0.06459824219334187, 

In [6]:
artifact = comparison.to_artifact()
source_manifest = {
    'path': 'data/manifests/nhanes-2017-2018-patient-need.json',
    'source': manifest['source'],
    'release': manifest['release'],
    'retrieved_at': manifest['retrieved_at'],
    'join_policy': (
        'Documented one-to-one within-cycle NHANES component join; '
        'no cross-source or cross-person linkage'
    ),
    'files': [
        {key: item[key] for key in ('name', 'url', 'sha256', 'bytes')}
        for item in manifest['files']
    ],
}
artifact.update({
    'evidence_type': evidence_type, 'source': source_label,
    'analysis_population': (
        'NHANES 2017-2018 MEC-examined adults excluding known pregnancy with complete eGFR and UACR'
        if evidence_type == 'public_observed' else 'Deterministic fixture only'
    ),
    'outcome': 'Cross-sectional eGFR <60 or UACR >=30 mg/g indicator; not confirmed CKD',
    'performance_interpretation': (
        'Unweighted held-out analytic-sample metrics; not population estimates'
    ),
    'source_manifest': source_manifest,
})
artifact_path, checksum_path = patient_finding_output_paths(ROOT / 'data', evidence_type)
artifact_path.parent.mkdir(parents=True, exist_ok=True)
payload = json.dumps(artifact, sort_keys=True, indent=2, allow_nan=False) + '\n'
artifact_path.write_text(payload, encoding='utf-8')
digest = sha256(payload.encode('utf-8')).hexdigest()
checksum_record = f'{digest}  {artifact_path.relative_to(ROOT).as_posix()}\n'
checksum_path.write_text(checksum_record, encoding='utf-8')
print({
    'artifact': str(artifact_path.relative_to(ROOT)),
    'sha256': digest,
    'contains_row_ids': False,
})


{'artifact': 'data/processed/patient_finding_model_comparison.json', 'sha256': '88e4cb69e54067bbcbea04e2c6b61bcdd6c6b6d4889ac92f3bc2cc9a4331baf8', 'contains_row_ids': False}


## Interpretation guardrails

- ROC-AUC measures ranking; PR-AUC is interpreted against the held-out prevalence/no-skill reference. Brier score and five equal-count calibration bins describe probability error.
- A reference threshold is selected from out-of-fold development predictions. For a fair operational comparison, each holdout model ranks probabilities without labels and selects exactly the requested capacity; ties resolve by stable row order, so selected count never exceeds capacity.
- Age-band and sex results are descriptive diagnostics, not fairness certification. The artifact reports denominators, events, and small-sample caveats. Race/ethnicity is included as a pre-lab demographic comparator but is not presented as a biological cause.
- The model does not use survey weights. Its performance applies to this analytic split only; Task 3 contains the survey-weighted population estimate. No causal, diagnostic, treatment, or individual targeting claim is supported.